# E-Commerce Customer Segmentation & Personalized Recommendation System

# Project Overview

This project groups e-commerce customers based on purchasing behavior and uses the segments to support personalized product recommendations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from segmentation_pipeline import (
    FEATURE_COLUMNS,
    RANDOM_STATE,
    load_raw_data,
    audit_raw_data,
    clean_transactions,
    build_customer_features,
    fit_preprocessor,
    evaluate_kmeans,
    evaluate_agglomerative,
    select_final_k,
    build_cluster_profiles,
    assign_personas,
    assign_new_customer,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
print(f'Libraries loaded. Random state: {RANDOM_STATE}')

## Dataset & Problem

The **Online Retail II** dataset contains ~525 k transaction rows from a UK online retailer (Dec 2009 – Dec 2010). Each row is one product line on an invoice. Cancellations are prefixed with 'C' and contain negative quantities.

## Data Audit

Load the raw CSV and inspect quality issues.

In [ ]:
raw_df = load_raw_data('online_retail_II.csv')
audit = audit_raw_data(raw_df)

print(f"Shape: {audit['rows']:,} rows × {len(audit['columns'])} columns")
print(f"Columns: {audit['columns']}")
print(f"Date range: {audit['date_min']} to {audit['date_max']}")
print(f"Customers: {audit['unique_customers']:,} | Invoices: {audit['unique_invoices']:,} | StockCodes: {audit['unique_stock_codes']:,}")
print(f"Missing values: {audit['missing_values']}")
print(f"Exact duplicates: {audit['exact_duplicates']:,}")
print(f"C-prefixed rows/invoices: {audit['cancelled_rows']:,}/{audit['cancelled_invoices']:,}")
print(f"Negative quantities: {audit['negative_quantity_rows']:,} (non-C: {audit['non_cancelled_negative_quantity_rows']:,})")
print(f"Zero/negative price rows: {audit['zero_price_rows']:,}/{audit['negative_price_rows']}")
print(f"Non-product-code rows/codes: {audit['non_product_rows']:,}/{audit['non_product_codes']}")

## Data Cleaning

Remove missing Customer IDs, duplicates, non-product stock codes, and zero-price rows. Separate purchases from returns/cancellations.

In [ ]:
purchases, returns, clean_all, cleaning_report = clean_transactions(raw_df)
print(pd.Series(cleaning_report).to_string())
print(f"Purchase date range: {purchases['InvoiceDate'].min()} to {purchases['InvoiceDate'].max()}")

## Customer Feature Engineering

Aggregate transaction rows into 8 customer-level features:
Recency, Frequency, Monetary, AOV, ProductVariety, ReturnRatio, AvgQtyPerOrder, ActiveSpan.

In [ ]:
customer_df = build_customer_features(purchases, returns)
print(f'Customer feature matrix: {customer_df.shape}')
customer_df.head(10)

## Preprocessing & Scaling

Clip outliers at 1st/99th percentiles, log-transform skewed features, then StandardScaler.

In [ ]:
preprocessor, X_scaled = fit_preprocessor(customer_df)
preprocessor.reference_date = purchases['InvoiceDate'].max() + pd.Timedelta(days=1)

print(f'Log-transformed features: {preprocessor.log_features}')
print()
bounds = pd.DataFrame({'p01': preprocessor.lower_bounds, 'p99': preprocessor.upper_bounds})
print('Fitted clipping bounds:')
print(bounds)
print()
print('Scaled feature summary:')
print(X_scaled.describe().loc[['mean', 'std']].round(1))

## Choosing the Number of Clusters

Evaluate K-Means (k = 2–10) using Silhouette, Calinski-Harabasz, and Davies-Bouldin scores. Compare against Ward Agglomerative clustering.

In [ ]:
kmeans_evaluation = evaluate_kmeans(X_scaled, random_state=RANDOM_STATE)
agglomerative_evaluation = evaluate_agglomerative(X_scaled)

print('=== K-Means evaluation ===')
print(kmeans_evaluation.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['silhouette', 'calinski_harabasz', 'davies_bouldin']):
    ax.plot(kmeans_evaluation.index, kmeans_evaluation[metric], 'o-', label='K-Means')
    ax.plot(agglomerative_evaluation.index, agglomerative_evaluation[metric], 's--', label='Ward')
    ax.set(xlabel='k', ylabel=metric.replace('_', ' ').title(), title=metric.replace('_', ' ').title())
    ax.legend()
plt.tight_layout()
plt.show()

## K-Means Clustering

Select k = 4 (verified by the 85 % silhouette threshold rule) and fit the final K-Means model.

In [ ]:
FINAL_K = select_final_k(kmeans_evaluation, preferred_k=4)
kmeans = KMeans(n_clusters=FINAL_K, init='k-means++', n_init=10, max_iter=300, random_state=RANDOM_STATE)
customer_df['Cluster'] = kmeans.fit_predict(X_scaled.values)

print(f'Selected k: {FINAL_K}')
print('Cluster sizes:', customer_df['Cluster'].value_counts().sort_index().to_dict())

print(f'\nComparison at k={FINAL_K}:')
comparison = pd.DataFrame({
    'K-Means': kmeans_evaluation.loc[FINAL_K, ['silhouette', 'calinski_harabasz', 'davies_bouldin']],
    'Ward Agglomerative': agglomerative_evaluation.loc[FINAL_K, ['silhouette', 'calinski_harabasz', 'davies_bouldin']],
})
print(comparison.round(4))

## Cluster Profiles & Personas

Summarise clusters on original (unscaled) median values and assign data-driven business personas.

In [ ]:
profile_median, profile_mean = build_cluster_profiles(customer_df)
personas = assign_personas(profile_median)
customer_df['Persona'] = customer_df['Cluster'].map({c: d['name'] for c, d in personas.items()})
persona_map = {c: d['name'] for c, d in personas.items()}

print('=== Median cluster profile ===')
print(profile_median.to_string())
print()
print('=== Personas ===')
for cluster, data in sorted(personas.items()):
    print(f"C{cluster}: {data['name']} ({data['customer_count']:,} customers) — {data['description']}")

actions = {
    'Champions': 'VIP rewards, early access, referral programmes',
    'At-Risk High-Value': 'Win-back campaigns and feedback requests',
    'Loyal Regulars': 'Cross-sell and loyalty offers',
    'Recent / Growing': 'Onboarding and product education',
    'Dormant / Lapsed': 'Re-engagement campaigns',
    'Occasional Buyers': 'Targeted promotions and bundles',
    'High-Return Risk': 'Quality and satisfaction investigation',
}
action_table = pd.DataFrame([
    {'Cluster': c, 'Persona': d['name'], 'Customers': d['customer_count'],
     'Action': actions.get(d['name'], 'Review')}
    for c, d in sorted(personas.items())
])
display(action_table)

## PCA Visualization

Project the 8-dimensional feature space to 2D using PCA to visualise cluster separation.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled.values)
pca_coordinates = pd.DataFrame(X_pca, columns=['PC1', 'PC2'], index=customer_df.index)
pca_coordinates['Cluster'] = customer_df['Cluster'].values

print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}')

fig, ax = plt.subplots(figsize=(10, 7))
for cluster in range(FINAL_K):
    mask = pca_coordinates['Cluster'] == cluster
    ax.scatter(pca_coordinates.loc[mask, 'PC1'], pca_coordinates.loc[mask, 'PC2'],
              s=18, alpha=0.45, label=f'C{cluster}: {persona_map[cluster]}')
centroids_2d = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_2d[:, 0], centroids_2d[:, 1], marker='X', c='black', s=160, label='Centroids')
ax.set(xlabel='PC1', ylabel='PC2', title=f'Customer segments — PCA projection (k={FINAL_K})')
ax.legend()
plt.tight_layout()
plt.show()

## Individual Radar Charts for the 4 Clusters

Normalised median profiles for each cluster, plotted individually.

In [ ]:
profile = profile_median[FEATURE_COLUMNS].copy()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
features = profile_norm.columns.tolist()

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

for cluster in range(FINAL_K):
    values = profile_norm.loc[cluster].values
    values = np.concatenate([values, [values[0]]])
    persona = personas[cluster]['name']

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, polar=True)
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(features, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_title(f'Cluster {cluster} — {persona}', pad=20, fontsize=14)
    plt.tight_layout()
    plt.show()

## New Customer Assignment

Predict the segment for a single customer using the fitted preprocessor and K-Means model.

In [ ]:
demo_customer_id = int(customer_df['CustomerID'].iloc[0])
demo_purchases = purchases[purchases['Customer ID'] == demo_customer_id]
demo_returns = returns[returns['Customer ID'] == demo_customer_id]

assignment = assign_new_customer(demo_purchases, demo_returns, preprocessor, kmeans, personas)
batch_cluster = int(customer_df.loc[customer_df['CustomerID'] == demo_customer_id, 'Cluster'].iloc[0])

print('Assignment result:')
print(assignment)
print(f'Batch cluster: {batch_cluster}; assignment cluster: {assignment["cluster"]}')
assert assignment['cluster'] == batch_cluster, 'Mismatch!'

## Recommendation System

Build a hybrid recommender:
- **Item similarity** — cosine similarity across customer–product purchase vectors
- **Segment popularity** — best-selling items within the customer's cluster
- **Cold-start fallback** — overall bestsellers for customers with no history

In [ ]:
customer_product = purchases.groupby(['Customer ID', 'StockCode'])['Quantity'].sum().unstack(fill_value=0)
item_similarity = cosine_similarity(customer_product.T)
item_sim_df = pd.DataFrame(item_similarity, index=customer_product.columns, columns=customer_product.columns)

print(f'Customer-product matrix: {customer_product.shape}; sparsity: {(customer_product == 0).to_numpy().mean():.2%}')
print(f'Item similarity matrix: {item_sim_df.shape}')

def get_cluster_top_products(purchases_df, segmented_customers, n_top=10):
    purchases_with_cluster = purchases_df.merge(
        segmented_customers[['CustomerID', 'Cluster', 'Persona']],
        left_on='Customer ID', right_on='CustomerID', how='inner'
    )
    top_by_cluster = {}
    for cluster in sorted(segmented_customers['Cluster'].unique()):
        cluster_purchases = purchases_with_cluster[purchases_with_cluster['Cluster'] == cluster]
        top_by_cluster[cluster] = cluster_purchases.groupby('StockCode').agg(
            total_qty=('Quantity', 'sum'),
            unique_buyers=('Customer ID', 'nunique'),
            description=('Description', lambda v: v.mode().iloc[0] if not v.mode().empty else 'N/A'),
        ).nlargest(n_top, 'unique_buyers')
    return top_by_cluster

cluster_top_products = get_cluster_top_products(purchases, customer_df)

def cold_start_recommendations(purchases_df, n_recommendations=5):
    top = purchases_df.groupby('StockCode').agg(
        unique_buyers=('Customer ID', 'nunique'),
        description=('Description', lambda v: v.mode().iloc[0] if not v.mode().empty else 'N/A'),
    ).nlargest(n_recommendations, 'unique_buyers')
    return pd.DataFrame({
        'StockCode': top.index,
        'Description': top['description'].values,
        'Score': (top['unique_buyers'] / top['unique_buyers'].max()).round(4).values,
        'Source': 'cold_start',
    })

def recommend_products(customer_id, purchases_df, segmented_customers, item_similarities, top_by_cluster, n_recommendations=5):
    cluster_info = segmented_customers[segmented_customers['CustomerID'] == customer_id]
    if cluster_info.empty:
        return cold_start_recommendations(purchases_df, n_recommendations)

    cluster = int(cluster_info['Cluster'].iloc[0])
    customer_purchases = purchases_df[purchases_df['Customer ID'] == customer_id]
    bought = set(customer_purchases['StockCode'].unique())
    similarity_scores = {}
    for product in bought:
        if product not in item_similarities.index:
            continue
        quantity_weight = customer_purchases.loc[customer_purchases['StockCode'] == product, 'Quantity'].sum()
        candidates = item_similarities[product].drop(index=bought.intersection(item_similarities.index), errors='ignore')
        for candidate, score in candidates.items():
            if score > 0:
                similarity_scores[candidate] = similarity_scores.get(candidate, 0) + score * np.log1p(quantity_weight)

    segment_scores = {}
    cluster_popular = top_by_cluster.get(cluster, pd.DataFrame())
    if not cluster_popular.empty:
        max_buyers = cluster_popular['unique_buyers'].max()
        segment_scores = {
            product: row['unique_buyers'] / max_buyers
            for product, row in cluster_popular.iterrows()
            if product not in bought
        }

    max_similarity = max(similarity_scores.values()) if similarity_scores else 1
    combined = {
        product: 0.7 * (similarity_scores.get(product, 0) / max_similarity) + 0.3 * segment_scores.get(product, 0)
        for product in set(similarity_scores).union(segment_scores)
    }
    rows = []
    for product, score in sorted(combined.items(), key=lambda pair: pair[1], reverse=True)[:n_recommendations]:
        description = purchases_df.loc[purchases_df['StockCode'] == product, 'Description'].mode()
        rows.append({
            'StockCode': product,
            'Description': description.iloc[0] if not description.empty else 'N/A',
            'Score': round(score, 4),
            'Source': 'hybrid',
        })
    return pd.DataFrame(rows)

## Recommendation Example

Hybrid recommendations for an existing customer and cold-start fallback for a new customer.

In [ ]:
sample_customer = int(customer_df['CustomerID'].iloc[0])
print('Hybrid recommendations for an existing customer:')
recommendations = recommend_products(sample_customer, purchases, customer_df, item_sim_df, cluster_top_products)
display(recommendations)
print('Cold-start recommendations:')
display(cold_start_recommendations(purchases))

# Visualise recommendation scores
plt.figure(figsize=(9, 5))
plt.barh(recommendations['Description'].astype(str), recommendations['Score'], color='#2b5c8f')
plt.xlabel('Recommendation Score')
plt.ylabel('Product')
plt.title(f'Top 5 Product Recommendations — Customer {sample_customer}')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Recommendation Evaluation

Offline temporal evaluation: split transactions into training (first 80 %) and test (last 20 %).
- **Hit Rate@5** — % of customers who received ≥ 1 relevant recommendation
- **Catalog Coverage** — % of training items that appear in recommendations

In [ ]:
date_range = purchases['InvoiceDate'].max() - purchases['InvoiceDate'].min()
split_date = purchases['InvoiceDate'].min() + date_range * 0.8
train_purchases = purchases[purchases['InvoiceDate'] < split_date]
test_purchases = purchases[purchases['InvoiceDate'] >= split_date]
train_cp = train_purchases.groupby(['Customer ID', 'StockCode'])['Quantity'].sum().unstack(fill_value=0)
train_item_sim = cosine_similarity(train_cp.T)
train_item_sim_df = pd.DataFrame(train_item_sim, index=train_cp.columns, columns=train_cp.columns)

# Convert to NumPy for fast evaluation
sim_matrix = train_item_sim_df.values
stockcode_to_idx = {code: i for i, code in enumerate(train_item_sim_df.index)}
idx_to_stockcode = list(train_item_sim_df.index)

overlap_customers = sorted(set(train_purchases['Customer ID']).intersection(test_purchases['Customer ID']))
eval_sample = overlap_customers[:500]
hits = total = 0
recommended_products = set()

train_bought_by_cust = train_purchases.groupby('Customer ID')['StockCode'].apply(set).to_dict()
test_bought_by_cust = test_purchases.groupby('Customer ID')['StockCode'].apply(set).to_dict()

for customer_id in eval_sample:
    train_bought = train_bought_by_cust.get(customer_id, set())
    test_bought = test_bought_by_cust.get(customer_id, set())
    new_purchases = test_bought - train_bought
    if not new_purchases:
        continue

    bought_idx = {stockcode_to_idx[p] for p in train_bought if p in stockcode_to_idx}
    scores = {}
    for product in train_bought:
        if product not in stockcode_to_idx:
            continue
        p_idx = stockcode_to_idx[product]
        sim_row = sim_matrix[p_idx].copy()
        for b_idx in bought_idx:
            sim_row[b_idx] = -1.0
        top_indices = np.argsort(sim_row)[-10:]
        for idx in top_indices:
            score = sim_row[idx]
            if score > 0:
                candidate = idx_to_stockcode[idx]
                scores[candidate] = scores.get(candidate, 0) + score

    recs = set(sorted(scores, key=scores.get, reverse=True)[:5])
    recommended_products.update(recs)
    hits += bool(recs.intersection(new_purchases))
    total += 1

hit_rate = hits / total if total else 0
coverage = len(recommended_products) / train_purchases['StockCode'].nunique()
print(f'Train/test rows: {len(train_purchases):,}/{len(test_purchases):,}; evaluated customers: {total}')
print(f'Item-similarity Hit Rate@5: {hit_rate:.2%}')
print(f'Training-catalog coverage: {coverage:.2%}')

## Final Insights

| Cluster | Persona | Customers | Recommended Action |
|---------|---------|-----------|--------------------|
| 0 | Champions | 1,449 | VIP rewards, early access, referral programmes |
| 1 | Occasional Buyers | 1,511 | Targeted promotions and bundles |
| 2 | Dormant / Lapsed | 1,218 | Re-engagement campaigns |
| 3 | High-Return Risk | 107 | Quality and satisfaction investigation |

The four customer segments show different purchasing behaviors and suggest different business strategies.

# Limitations & Future Improvements

The project uses one retailer's historical dataset, so the results may not generalize to other businesses.

Future improvements could include more data, stronger recommendation evaluation, and regular model updates as customer behavior changes.